# `Semana 8`: Programación Funcional

---
## 1. Paradigma Funcional

Python es un lenguaje multiparadigma: se puede programar de forma procedimental, orientada a objetos o **funcional**, incluso combinando enfoques dentro de un mismo programa.

En **programación funcional**, una función se comporta como una función matemática: dado el mismo *input* siempre retorna el mismo *output*, y el resultado depende `únicamente` de sus `parámetros de entrada`.

Una función es **pura** cuando:
- Retorna siempre el mismo *output* para el mismo *input*.
- No produce **efectos secundarios** (*side effects*): no modifica nada fuera de su propio *scope* (variables externas, prints, archivos, etc.).

Para verificar si una función es pura:
1. El *output* depende solo del *input*.
2. La función deja todo lo demás sin modificar.

Si una de las dos no se cumple, la función es `impura`.

In [17]:
from typing import Any
from copy import deepcopy

lista = ["Hola", True, 3]

def impura(elemento : Any) -> Any:
    lista.append(elemento)
    return elemento

# La anterior función es impura, pues, aunque dado el mismo
# input, se retorna el mismo output, hay efectos secundarios
# al modificar una lista externa.

def pura(elemento : Any, lista : list) -> list:
    lista_copia = deepcopy(lista)
    lista_copia.append(elemento)
    return lista_copia

# A diferencia de la función "impura",
# se crea una copia de la lista, y luego 
# se le modifica. Con esto, dado el mismo input,
# se obtendrá siempre el mismo output, y 
# no hay efectos secundarios, al no modificar 
# una lista externa.

print("= Output de funciones puras =")
print(pura(False, lista))
print(pura("FRNND4", lista))
print(f"Lista original: {lista}")
print("="*29)

impura(False)
impura("FRNND4")
impura("FRNND4")
impura("FRNND4")

print("= Output de funciones impuras =")
print(impura(False))
print(impura("FRNND4"))
print(impura("FRNND4"))
print(impura("FRNND4"))
print(f"Lista original: {lista}")

= Output de funciones puras =
['Hola', True, 3, False]
['Hola', True, 3, 'FRNND4']
Lista original: ['Hola', True, 3]
= Output de funciones impuras =
False
FRNND4
FRNND4
FRNND4
Lista original: ['Hola', True, 3, False, 'FRNND4', 'FRNND4', 'FRNND4', False, 'FRNND4', 'FRNND4', 'FRNND4']


-  Al no modificar nada externo, un cambio inesperado en un valor no puede venir de una función pura.
-  Como el *output* solo depende del *input*, se puede cachear el resultado y evitar recalcularlo.
-  Al no haber efectos secundarios, los flujos de datos (*data flow*) son más simples de razonar y de iterar.

In [35]:
from functools import cache

# Creamos la función pura tribonacci, y con
# el decorador @cache de functools, podemos
# almacenar las llamadas para no calcularlas
# nuevamente (requiere argumentos hasheables
# y funciones deterministas)

@cache
def tribonacci(num: int) -> int:
    if num == 0:
        return 0
    elif num == 1 or num == 2:
        return 1
    return tribonacci(num - 1) + tribonacci(num - 2) + tribonacci(num - 3)

print(tribonacci(25))  

1389537


---
## 2. Funciones Generadoras

Un **generador**, como recordatorio, permite iterar sobre una secuencia sin almacenar todos sus elementos en memoria: los produce sobre la marcha. 

Una función se convierte en **función generadora** si contiene la sentencia `yield` en algún punto de su cuerpo.

- `yield` es similar a `return`, pero **pausa** la ejecución de la función guardando su estado, en vez de terminarla (cada vez que se itera un generador, se reanuda dicha función).
- Al llamar a una función generadora **no se ejecuta su código**: solo se crea un objeto generador.
- El generador es su propio iterador (`__iter__` retorna `self`), por lo que se puede recorrer con `for` o avanzar manualmente con `next()`.
- Un generador **solo se puede recorrer una vez**; al agotarse, desaparece.
- Una función generadora retorna generadores; se le tiene que asignar a una variable (para poder reutilizarse).

In [69]:
from typing import Generator
# Creamos una función generadora infinita que cada vez que se itera,
# produce un número par siguiente al anterior.

def generador_pares() -> Generator:
    par_actual = 0
    
    while True:
        yield par_actual
        par_actual += 2
        
pares = generador_pares()

lista_pares = []

for i in range(10):
    # Iteramos pares con next()
    lista_pares.append(next(pares))
    
print(lista_pares)

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


In [68]:
for i in range(6):
    print(next(pares))
    
print("El generador reanuda en donde se había quedado.")

20
22
24
26
28
30
El generador reanuda en donde se había quedado.


### `send()` y `yield from`

- `send(valor)`: Reanuda el generador enviándole un `valor`, que la expresión `yield` recibirá. Antes de poder usar `send`, hay que avanzar el generador una vez (con `next()`) hasta el primer `yield`.
- `yield from <iterable>`: Permite crear un generador a partir de otro iterable (u otro generador) sin necesidad de un `for`/`while` explícito. Es muy útil para generadores compuestos o anidados.

In [163]:
# Simularemos una caja para almacenar dinero con send() y la función
# generadora guardar_dinero

def guardar_dinero() -> Generator:
    
    dinero_actual = 0
    
    while True:
        # Aquí, se produce dinero_actual y 
        # se recibe con send() dinero_recibido.
        # Izquierda : Lo que se recibe.
        # Derecha: Lo que se produce (retorna).
        dinero_recibido = yield dinero_actual
        
        print(f"Hemos recibido {dinero_recibido} 🤑")
        
        if type(dinero_recibido) not in [int, float]:
            dinero_recibido = 0
            
        # Sumamos el dinero a la caja actual
        dinero_actual += dinero_recibido
        print(f"Sumamos ${dinero_recibido} a la caja, teniendo ahora: ${dinero_actual}\n")
        
# Creamos el generador
caja_1 = guardar_dinero()

# No se le puede enviar con send todavía,
caja_1.send(1)

TypeError: can't send non-None value to a just-started generator

In [164]:
next(caja_1)

# Ahora, podemos ir inyectándole valores para ir
# sumándole a la caja.

caja_1.send(100)
caja_1.send(200)

valor_actual_caja = caja_1.send(None)

print(f"Cantidad de dinero actual: {valor_actual_caja}")

Hemos recibido 100 🤑
Sumamos $100 a la caja, teniendo ahora: $100

Hemos recibido 200 🤑
Sumamos $200 a la caja, teniendo ahora: $300

Hemos recibido None 🤑
Sumamos $0 a la caja, teniendo ahora: $300

Cantidad de dinero actual: 300


In [162]:
# Con yield from, retornamos los elementos de un iterable,
# básicamente, se utiliza next en cada iteración del generador.

numeros_primos = [2, 3, 5, 7, 11, 13, 17]

def generador_primos_1() -> Generator:
    for num in numeros_primos:
        yield num
    
def generador_primos_2() -> Generator:
    yield from numeros_primos

print(" >>> Probando generador sin yield from.")
generador_1 = generador_primos_1()

try:
    for i in range(10):
        print(next(generador_1))
        
except StopIteration:
    print(f">>> Se ha consumido el generador en la iteración {i}.\n")
    
print(">>> Probando generador con yield from.")
generador_2 = generador_primos_2()

try:
    for i in range(10):
        print(next(generador_2))
        
except StopIteration:
    print(f">>> Se ha consumido el generador en la iteración {i}.")

 >>> Probando generador sin yield from.
2
3
5
7
11
13
17
>>> Se ha consumido el generador en la iteración 7.

>>> Probando generador con yield from.
2
3
5
7
11
13
17
>>> Se ha consumido el generador en la iteración 7.


---
## 3. Funciones que Trabajan con Iterables

Las funciones que **necesitan una función** como argumento:

- `map(f, *iterables)`: Aplica func a cada elemento del iterable y retorna un iterador con los resultados. Si se entregan varios iterables, func debe aceptar tantos parámetros como iterables se pasen, y el resultado tiene el largo del iterable más corto. Es similar a (func(x) for x in iterable).

- `filter(func, iterable)`: Retorna un generador con los elementos del iterable para los que func retorna True. Equivale a (x for x in iterable if func(x)).

- `reduce(func, iterable, inicializador=None)`: Aplica func acumulativamente de a pares (solo de a pares): func(func(func(a, b), c), d)..., hasta reducir el iterable a un solo valor. Si el iterable está vacío y no hay inicializador, lanza TypeError.

- `Funciones lambda`: Son funciones anónimas de una sola expresión: lambda parámetros: expresión. No necesitan def ni return, y son muy usadas junto a map, filter y reduce. Al ser anónimas, no se almacenan a menos que se les declare con una variable como funcion = lambda parámetros : expresión.

In [ ]:
from collections import namedtuple
from functools import reduce

Persona = namedtuple(
    "Persona",
    ["id", "nombre", "profesion", "sueldo", "antiguedad_anios", "activo"],
)

# Buscaremos las personas que son de profesión Data Scientist, y sumaremos sus sueldos.
personas = [
    Persona(101, "Fernanda", "Data Scientist", 5000, 5, True),
    Persona(102, "Bruno", "Diseñador", 2500, 1, False),
    Persona(103, "Carla", "Desarrolladora", 4800, 4, True),
    Persona(104, "Daniel", "QA", 2100, 3, True),
    Persona(105, "Isidora", "DevOps", 5200, 6, True),
    Persona(106, "Federico", "Diseñador", 3000, 2, True),
    Persona(107, "Gloria", "QA", 2300, 4, False),
    Persona(108, "Hugo", "Desarrolladora", 3900, 3, True),
    Persona(109, "Diego", "Data Scientist", 5800, 5, True),
    Persona(110, "Javier", "Backend", 4200, 1, True),
    Persona(111, "Antonia", "Frontend", 3500, 3, True),
    Persona(112, "Luis", "DevOps", 4900, 0, False),
    Persona(113, "Amalia", "Data Scientist", 6100, 7, True),
    Persona(114, "Nicolás", "QA", 2700, 3, True),
    Persona(115, "Javiera", "Diseñadora", 3100, 4, True),
    Persona(116, "Pablo", "Backend", 4500, 2, False),
    Persona(117, "Benjamín", "Frontend", 3300, 5, True),
    Persona(118, "Rosa", "Desarrolladora", 4600, 3, True),
    Persona(119, "Sergio", "DevOps", 5400, 1, False),
    Persona(120, "Dante", "Data Scientist", 6000, 4, True),
]

# Filtramos utilizando lambda y profesion == "Data Scientist" en filter.
# Siendo personas_filtradas un generador:
personas_filtradas = filter(lambda persona: persona.profesion == "Data Scientist", personas)

# Obtenemos los sueldos utilizando lambda y map.
sueldos_data_scientist = map(lambda persona: persona.sueldo, personas_filtradas)

suma = reduce(lambda acumulado, sueldo_actual: acumulado + sueldo_actual, sueldos_data_scientist)


print(suma)

22900


Las funciones que **no necesitan una función** como argumento:
- `enumerate(iterable)`: Entrega tuplas (indice, elemento), evitando tener que iterar manualmente con range(len(...)).

- `zip(*iterables)`: Combina varios iterables en tuplas con los elementos i-ésimos de cada uno. El resultado tiene el largo del iterable más corto. Usando zip(*zipped) se puede "des-zippear" una lista de tuplas.

In [194]:

SensorLog = namedtuple("SensorLog", ["timestamp_sec", "temperatura", "estado"])

# Queremos crear una lista de namedtuples,
# las cuales tomen la discrepancia entre ambos sensores,
# teniendo en cuenta de que el índice i en ambos registros
# corresponde al mismo registro.

registros_a = [
    SensorLog(0, 21.5, "OK"),
    SensorLog(5, 22.0, "OK"),
    SensorLog(10, 45.0, "WARNING"),
    SensorLog(15, 23.1, "OK"),
    SensorLog(20, 24.0, "OK"),
    SensorLog(25, 50.0, "CRITICAL"),
]

registros_b = [
    SensorLog(0, 21.3, "OK"),
    SensorLog(5, 21.8, "OK"),
    SensorLog(10, 46.1, "WARNING"),
    SensorLog(15, 23.5, "OK"),
    SensorLog(20, 24.2, "OK"),
    SensorLog(25, 51.5, "CRITICAL"),
]

registros_discrepancias = []

DiscrepanciaLog = namedtuple("DiscrepanciaLog", ["indice", "discrepancia", "alerta"])

# Siendo alerta True si es que la discrepancia entre ambos sensores es
# mayor a 1.

# Creamos el iterador que contiene las tuplas de ambos registros
# por índice.
registros_zip = zip(registros_a, registros_b)

# Con enumerate, creamos el iterador que contiene
# la tupla de indice y registros.
for indice, registros in enumerate(registros_zip):
    registro_1, registro_2 = registros
    
    discrepancia = round(abs(registro_1.temperatura - registro_2.temperatura), 2)
    alerta = True if discrepancia >= 1 else False
    
    log_actual = DiscrepanciaLog(indice, discrepancia, alerta)
    registros_discrepancias.append(log_actual)
    
print("== Discrepancia entre sensores del mismo registro ==")
for log in registros_discrepancias:
    print(log)
print("="*52)



== Discrepancia entre sensores del mismo registro ==
DiscrepanciaLog(indice=0, discrepancia=0.2, alerta=False)
DiscrepanciaLog(indice=1, discrepancia=0.2, alerta=False)
DiscrepanciaLog(indice=2, discrepancia=1.1, alerta=True)
DiscrepanciaLog(indice=3, discrepancia=0.4, alerta=False)
DiscrepanciaLog(indice=4, discrepancia=0.2, alerta=False)
DiscrepanciaLog(indice=5, discrepancia=1.5, alerta=True)


---
## 4. Aplicaciones: Funciones y Librerías *built-in*

`reversed(secuencia)`: Retorna una copia de la secuencia en orden inverso. Se puede personalizar sobreescribiendo __reversed__ en una clase propia.

#### Otras funciones *built-in* sobre iterables
- Enfoque matemático: `sum()`, `min()`, `max()`.
- Enfoque booleano: `all()`, `any()`.

In [ ]:
# Podemos ver el largo de un generador
generador = (i for i in range(5))

largo = sum(1 for _ in generador)

print(f">>> Largo del generador: {largo}\n")

# Verificamos que todos los elementos de la lista sean de tipo int

lista = [3, 98789, 55, "Hola"]

# Vemos si todos son int
todos_int = all(map(lambda x: type(x) == int, lista))
print(f"Son todos int: {todos_int}")

algun_str = any(map(lambda x: type(x) == str, lista))
print(f"Hay algun str: {algun_str}\n")

lista = [10, 20, 30, 40, 50]

print(f"Mínimo de la lista: {min(lista)}")
print(f"Suma de todos los elementos de la lista: {sum(lista)}")
print(f"Máximo de la lista: {max(lista)}")

>>> Largo del generador: 5

Son todos int: False
Hay algun str: True

Mínimo de la lista: 10
Suma de todos los elementos de la lista: 150
Máximo de la lista: 50


### Librerías *built-in*: `itertools` y `collections`

- **`itertools`**: funciones especializadas para iteradores.
  - Iteradores infinitos: count(), cycle(), repeat().
  - Combinatoria: product(), permutations(), combinations().
- **`collections`**: alternativas a dict, list, set, tuple.
  - namedtuple, deque, defaultdict, y Counter (cuenta ocurrencias de elementos hasheables en un iterable, incluye most_common(n)).

In [244]:
from collections import Counter
from itertools import combinations, count, cycle, permutations, product

# Probamos los iteradores infinitos
print(">>> count(10, 2)")
# count(inicio, step)
for i in count(10, 2):
    print(i, end=" ")
    if i >= 18:
        print()
        break
print()

print('>>> cycle(["ROJO", "AMARILLO", "VERDE"])')
semaforo = cycle(["ROJO", "AMARILLO", "VERDE"])
for _ in range(5):
    print(next(semaforo), end=" -> ")
print("...\n")

# Combinatoria
items = ["X", "Y"]
print("# Product (producto cartesiano):")
print(">>> list(product(items, repeat=2))")
print(list(product(items, repeat=2)), end="\n\n")

print("# Permutations (permutaciones):")
print('>> list(permutations("ABC", 2))')
print(list(permutations("ABC", 2)), end="\n\n")

print("# Combinatios (combinaciones):")
print('>> list(combinations("ABC", 2))')
print(list(combinations("ABC", 2)), end="\n\n")

# Counter
votos = ["rojo", "azul", "rojo", "verde", "rojo", "azul"]
conteo = Counter(votos)
print("Counter diccionario:\n   ", conteo)
print("Más común (top 1):\n   ", conteo.most_common(1))

>>> count(10, 2)
10 12 14 16 18 

>>> cycle(["ROJO", "AMARILLO", "VERDE"])
ROJO -> AMARILLO -> VERDE -> ROJO -> AMARILLO -> ...

# Product (producto cartesiano):
>>> list(product(items, repeat=2))
[('X', 'X'), ('X', 'Y'), ('Y', 'X'), ('Y', 'Y')]

# Permutations (permutaciones):
>> list(permutations("ABC", 2))
[('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]

# Combinatios (combinaciones):
>> list(combinations("ABC", 2))
[('A', 'B'), ('A', 'C'), ('B', 'C')]

Counter diccionario:
    Counter({'rojo': 3, 'azul': 2, 'verde': 1})
Más común (top 1):
    [('rojo', 3)]


**Cuándo usar generadores o functional tools**  
-> Cuando se trabaja con flujos continuos, archivos grandes o se quiere evitar crear listas temporales.

**Cuándo usar estructuras como list, Counter, diccionarios, etc**  
-> Cuando se necesita acceso aleatorio, conteos frecuentes, o recorrer la misma secuencia múltiples veces sin agotar el iterador.

1. Eficiencia
* **Cargar archivos**: Línea por línea (for linea in archivo) con generadores (menor consumo) vs cargar todo de golpe (readlines()).
* **Manejar información**: namedtuple (menor huella de memoria).

2. Comportamiento
* **Qué hacen `map`, `filter`, `itertools`**: Transforman, filtran y encadenan lógicas.
* **Qué retornan**: Iteradores/generadores vs estructuras concretas materializadas en la RAM (como listas, diccionarios, etc).


## Guía para elegir herramientas

| Situación | Herramienta |
| :--- | :--- |
| Procesar un archivo gigantesco de texto | `for line in file:` + generador |
| Contar frecuencias de elementos hasheables | `collections.Counter` |
| Encadenar filtros/mapeos sin gastar memoria intermedia | `filter()`, `map()`, generator expressions |
| Rerecorrer el resultado o indexar por posición | `list` / `tuple` |
| Definir registros planos de solo lectura livianos | `namedtuple` |